# EcoGrid AI — Demostración Integral de Modelos de IA y Minería de Texto
### Universidad Tecnológica de Calvillo (UTC) — Ingeniería en Tecnologías (7mo Cuatrimestre)

Este notebook integra las dos asignaturas de Inteligencia Artificial del proyecto:
1. **Fundamentos de IA**: Algoritmos supervisados (Redes Neuronales MLP, KNN, SVM) y no supervisados (K-Means, SOM) optimizados mediante Algoritmos Genéticos para clasificar patrones de consumo eléctrico.
2. **Minería de Texto**: Clasificación de incidentes y bitácoras eléctricas no estructuradas mediante NLP (TF-IDF + Clasificador) con reporte de **Accuracy** y **F1-Score**.

In [ ]:
import sys
import os
import numpy as np
from sklearn.model_selection import train_test_split

# Añadir directorio raíz al path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

from ai_engine.genetic.genetic_optimizer import GeneticHyperparameterOptimizer
from ai_engine.models.consumption_classifier import (
    EcoGridConsumptionPredictor,
    generate_synthetic_electrical_data,
    CLASS_NAMES
)
from ai_engine.models.text_mining_classifier import IncidentTextMiningClassifier

## 1. Fundamentos de IA: Generación de Datos de Telemetría Eléctrica
Generamos un conjunto de datos multivariable con parámetros eléctricos: Voltaje RMS, Corriente RMS, Potencia Activa, Potencia Reactiva, Factor de Potencia, Frecuencia, THD y Temperatura.

In [ ]:
X, y = generate_synthetic_electrical_data(n_samples=2000, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Distribución de clases: {np.bincount(y)}")
for idx, name in CLASS_NAMES.items():
    print(f"  Clase {idx}: {name}")

## 2. Optimización de Hiperparámetros con Algoritmo Genético (GA)
Ejecución de selección por torneo, cruzamiento uniforme y mutación para encontrar los mejores parámetros de KNN, SVM y Red Neuronal.

In [ ]:
ga_knn = GeneticHyperparameterOptimizer(model_type='knn', population_size=8, generations=4, random_state=42)
best_knn_params, score_knn = ga_knn.optimize(X_train, y_train, verbose=False)
print(f"Mejor configuración KNN: {best_knn_params} | Score CV: {score_knn:.4f}")

ga_svm = GeneticHyperparameterOptimizer(model_type='svm', population_size=8, generations=4, random_state=42)
best_svm_params, score_svm = ga_svm.optimize(X_train, y_train, verbose=False)
print(f"Mejor configuración SVM: {best_svm_params} | Score CV: {score_svm:.4f}")

## 3. Entrenamiento y Evaluación de Modelos (Supervisados y No Supervisados)
Se entrenan los modelos supervisados (KNN, SVM, MLP) y no supervisados (K-Means, SOM).

In [ ]:
predictor = EcoGridConsumptionPredictor()
results = predictor.fit_and_evaluate(X_train, y_train, X_test, y_test, ga_params={
    'knn': best_knn_params,
    'svm': best_svm_params
})

for model_name, metrics in results.items():
    print(f"=== Modelo: {model_name.upper()} ===")
    print(f"Exactitud (Accuracy): {metrics['accuracy'] * 100:.2f}%")
    print(f"F1-Score (Weighted): {metrics['f1_score']:.4f}")
    print("Matriz de Confusión:")
    print(np.array(metrics['confusion_matrix']))
    print()

## 4. Minería de Texto: Clasificación de Severidad de Bitácoras de Mantenimiento
Entrenamiento del Pipeline NLP (TF-IDF + Clasificador) y evaluación mediante Accuracy y F1-score.

In [ ]:
text_clf = IncidentTextMiningClassifier()
text_metrics = text_clf.train_and_evaluate(test_size=0.25, random_state=42)

print(f"Exactitud Global (Accuracy): {text_metrics['accuracy'] * 100:.2f}%")
print(f"F1-Score (Macro):            {text_metrics['f1_score_macro']:.4f}")
print(f"F1-Score (Ponderado):        {text_metrics['f1_score_weighted']:.4f}")
print("\nReporte por Clase de Severidad:")
for severity, scores in text_metrics['classification_report'].items():
    if isinstance(scores, dict):
        print(f"- {severity:10}: Precision={scores['precision']:.2f}, Recall={scores['recall']:.2f}, F1={scores['f1-score']:.2f}")

## 5. Prueba Interactiva de Incidentes No Estructurados

In [ ]:
test_incidents = [
    "Se detectó olor a quemado y chispas en barraje de 480V con humo denso",
    "Mantenimiento preventivo mensual de rutina y cambio de foco piloto",
    "Temperatura excesiva a 98 grados Celsius en conexión de fase B del alimentador",
    "Zumbido y vibración moderada en arrancador suave de bomba secundaria"
]

for inc in test_incidents:
    res = text_clf.classify_log(inc)
    print(f"Texto: '{inc}'")
    print(f" -> Severidad Predicha: {res['predicted_severity']} (Confianza: {res['confidence'] * 100:.1f}%)")
    print(f" -> Acción Recomendada: {res['recommended_action']}")
    print(f" -> Desconexión de Cargas Requerida: {res['requires_shutdown']}\n")